# EDA — Give Me Some Credit + simulated behavioral features

Short exploration to motivate the modelling choices:
1. How dirty is the raw data (missing values, sentinel codes, outliers)?
2. How big is the **thin-file** segment and how risky is it?
3. Do the **simulated** behavioral features carry a mild (not overwhelming) signal?

> The behavioral features are synthetic — see `src/simulate.py`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # run from notebooks/

import matplotlib.pyplot as plt
import pandas as pd
from src.config import BEHAVIORAL_FEATURES, TARGET, THIN_FILE_MAX_OPEN_LINES
from src.data import load_raw, clean
from src.features import thin_file_mask
from src.simulate import simulate_behavioral

raw = load_raw()
print(raw.shape)
raw.head()

## 1. Data quality

In [ ]:
quality = pd.DataFrame({
    "missing_%": raw.isna().mean().mul(100).round(2),
    "max": raw.max(),
    "p99": raw.quantile(0.99),
})
quality

In [ ]:
# Past-due counts of 96/98 are sentinel codes, not real counts
sentinel = raw["past_due_30_59"].isin([96, 98])
print(f"sentinel rows: {sentinel.sum()} | default rate among them: {raw.loc[sentinel, TARGET].mean():.1%}")
print(f"overall default rate: {raw[TARGET].mean():.2%}")

Sentinel rows default far more than average, so `src/data.py` maps them to the cap ("many late payments") instead of dropping them. Utilisation and debt ratio have extreme tails, hence the fixed caps.

## 2. Thin-file segment

In [ ]:
df = simulate_behavioral(clean(raw))
thin = thin_file_mask(df)
print(f"thin-file (<= {THIN_FILE_MAX_OPEN_LINES} open lines): {thin.mean():.1%} of applicants")
df.groupby(df["open_credit_lines"].clip(upper=15))[TARGET].agg(["mean", "size"]).rename(
    columns={"mean": "default_rate", "size": "n"}).T.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
rate = df.groupby(df["open_credit_lines"].clip(upper=20))[TARGET].mean()
ax.bar(rate.index, rate.values, color=["#eb6834" if i <= THIN_FILE_MAX_OPEN_LINES else "#2a78d6" for i in rate.index])
ax.set_xlabel("Open credit lines / loans (20 = 20+)"); ax.set_ylabel("Default rate")
ax.set_title("Default rate by number of open credit lines (orange = thin-file)", loc="left")
plt.show()

Thin-file applicants are both rarer and riskier — and the bureau has less information to separate good from bad among them. That is exactly where additional data should help most.

## 3. Simulated behavioral features

In [ ]:
df.groupby(TARGET)[BEHAVIORAL_FEATURES].mean().T.round(2)

In [ ]:
fig, axes = plt.subplots(1, len(BEHAVIORAL_FEATURES), figsize=(16, 3))
for ax, col in zip(axes, BEHAVIORAL_FEATURES):
    for y, color, label in ((0, "#2a78d6", "repaid"), (1, "#eb6834", "defaulted")):
        ax.hist(df.loc[df[TARGET] == y, col], bins=30, density=True, alpha=0.55, color=color, label=label)
    ax.set_title(col, fontsize=9, loc="left")
axes[0].legend()
plt.tight_layout(); plt.show()

Distributions overlap heavily: each feature shifts only slightly between classes, by design. Any lift comes from combining several weak signals, not from a leaked label.